In [5]:
import pandas as pd
df = pd.read_csv("sim.csv")

In [6]:
from collections import defaultdict
# Keep only necessary columns to reduce memory
df = df[['nameOrig', 'nameDest', 'amount', 'isFraud']]

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (6362620, 4)
      nameOrig     nameDest    amount  isFraud
0  C1231006815  M1979787155   9839.64        0
1  C1666544295  M2044282225   1864.28        0
2  C1305486145   C553264065    181.00        1
3   C840083671    C38997010    181.00        1
4  C2048537720  M1230701703  11668.14        0


In [7]:
# Get all unique accounts
accounts = sorted(set(df['nameOrig']) | set(df['nameDest']))

# Map account IDs to indices (required for Union-Find)
account_to_idx = {acc: i for i, acc in enumerate(accounts)}

print("Total unique accounts:", len(accounts))

Total unique accounts: 9073900


In [8]:
graph = defaultdict(list)

for _, row in df.iterrows():
    sender = row['nameOrig']
    receiver = row['nameDest']
    amount = row["amount"] 
    graph[sender].append((receiver, amount))

print("Graph built with nodes:", len(graph))

Graph built with nodes: 6353307


In [9]:
class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size
        self.size = [1] * size

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]

    def union(self, x, y):
        rootX = self.find(x)
        rootY = self.find(y)

        if rootX == rootY:
            return

        # Union by rank
        if self.rank[rootX] < self.rank[rootY]:
            self.parent[rootX] = rootY
            self.size[rootY] += self.size[rootX]
        elif self.rank[rootX] > self.rank[rootY]:
            self.parent[rootY] = rootX
            self.size[rootX] += self.size[rootY]
        else:
            self.parent[rootY] = rootX
            self.rank[rootX] += 1
            self.size[rootX] += self.size[rootY]

In [10]:
uf = UnionFind(len(accounts))

for _, row in df.iterrows():
    sender = row['nameOrig']
    receiver = row['nameDest']

    idx1 = account_to_idx[sender]
    idx2 = account_to_idx[receiver]

    uf.union(idx1, idx2)

In [11]:
components = defaultdict(list)

for acc in accounts:
    idx = account_to_idx[acc]
    root = uf.find(idx)
    components[root].append(acc)

print("Total clusters detected:", len(components))

Total clusters detected: 2711280


In [12]:
# Sort clusters by size
cluster_size = {}
for root, members in components.items():
    size = len(members)
    for acc in members:
        cluster_size[acc] = size

top5 = sorted(cluster_size.items(), key=lambda x: x[1], reverse=True)[:15]
print("Top 5 cluster sizes:")
for acc, size in top5:
    print(f"Account {acc} belongs to a cluster of size {size}")

Top 5 cluster sizes:
Account C1005754692 belongs to a cluster of size 121
Account C1007091734 belongs to a cluster of size 121
Account C1019533814 belongs to a cluster of size 121
Account C1027077713 belongs to a cluster of size 121
Account C1036327038 belongs to a cluster of size 121
Account C105198810 belongs to a cluster of size 121
Account C1054591790 belongs to a cluster of size 121
Account C1085158170 belongs to a cluster of size 121
Account C1119147995 belongs to a cluster of size 121
Account C1149226671 belongs to a cluster of size 121
Account C1159710789 belongs to a cluster of size 121
Account C1165424674 belongs to a cluster of size 121
Account C1171146548 belongs to a cluster of size 121
Account C1223487299 belongs to a cluster of size 121
Account C12333931 belongs to a cluster of size 121


In [13]:
#degree and aggregate statistics
in_degree = defaultdict(int)
out_degree = defaultdict(int)
total_sent = defaultdict(float)
total_received = defaultdict(float)
transaction_count = defaultdict(int)

for _, row in df.iterrows():
    sender, receiver = row['nameOrig'], row['nameDest']
    amt = row['amount']
    
    out_degree[sender] += 1
    in_degree[receiver] += 1
    total_sent[sender] += amt
    total_received[receiver] += amt
    transaction_count[sender] += 1
    transaction_count[receiver] += 1

In [27]:
#page rank
import networkx as nx

G = nx.DiGraph()

for _, row in df.iterrows():
    G.add_edge(row['nameOrig'], row['nameDest'])
pagerank = nx.pagerank(G, alpha=0.85)

In [43]:
# funnel
receiver_unique_senders = df.groupby('nameDest')['nameOrig'].nunique()

In [44]:
features = pd.DataFrame({
    'account': accounts,
    'in_degree': [in_degree.get(acc, 0) for acc in accounts],
    'out_degree': [out_degree.get(acc, 0) for acc in accounts],
    'total_sent': [total_sent.get(acc, 0) for acc in accounts],
    'total_received': [total_received.get(acc, 0) for acc in accounts],
    'pagerank': [pagerank.get(acc, 0) for acc in accounts],
    'receiver_unique_senders': [receiver_unique_senders.get(acc, 0) for acc in accounts]  
})
 
features['sent_received_ratio'] = (
    features['total_sent'] / (features['total_received'] + 1)
)
 

In [45]:
features.head()

,account,in_degree,out_degree,total_sent,total_received,pagerank,receiver_unique_senders,sent_received_ratio
0,C1000000639,0,1,244486.46,0.0,4.461723e-08,0,244486.46
1,C1000001337,0,1,3170.28,0.0,4.461723e-08,0,3170.28
2,C1000001725,0,1,8424.74,0.0,4.461723e-08,0,8424.74
3,C1000002591,0,1,261877.19,0.0,4.461723e-08,0,261877.19
4,C1000003372,0,1,20528.65,0.0,4.461723e-08,0,20528.65


In [46]:
df['sender_out_deg'] = df['nameOrig'].map(out_degree)
df['receiver_in_deg'] = df['nameDest'].map(in_degree) 
df['sender_cluster_size'] = df['nameOrig'].map(cluster_size)
df['receiver_cluster_size'] = df['nameDest'].map(cluster_size) 
df['sender_ratio'] = df['nameOrig'].map(features.set_index('account')['sent_received_ratio']) 
df['receiver_ratio'] = df['nameDest'].map(features.set_index('account')['sent_received_ratio']) 
df['receiver_funnel'] = df['nameDest'].map(receiver_unique_senders)
df['sender_pagerank'] = df['nameOrig'].map(pagerank)
df['receiver_pagerank'] = df['nameDest'].map(pagerank)

In [47]:
df.to_csv("final_map feature transactions.csv", index=False)

In [48]:
features.to_csv("features.csv", index=False)

In [49]:
df.shape

(6362620, 13)

In [50]:
features.shape

(9073900, 8)

In [51]:
df.head()

,nameOrig,nameDest,amount,isFraud,sender_out_deg,receiver_in_deg,sender_cluster_size,receiver_cluster_size,sender_ratio,receiver_ratio,receiver_funnel,sender_pagerank,receiver_pagerank
0,C1231006815,M1979787155,9839.64,0,1,1,2,2,9839.64,0.0,1,4.461723e-08,1.382925e-07
1,C1666544295,M2044282225,1864.28,0,1,1,2,2,1864.28,0.0,1,4.461723e-08,1.382925e-07
2,C1305486145,C553264065,181.00,1,1,44,49,49,181.00,0.0,44,4.461723e-08,4.119491e-06
3,C840083671,C38997010,181.00,1,1,41,42,42,181.00,0.0,41,4.461723e-08,3.885303e-06
4,C2048537720,M1230701703,11668.14,0,1,1,2,2,11668.14,0.0,1,4.461723e-08,1.382925e-07


## MODEL TRAINING

In [52]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [53]:
x= df.drop(columns=['isFraud', 'nameOrig', 'nameDest'])
y = df['isFraud']

In [58]:
# target variable is skewed.

In [57]:
df['isFraud'].value_counts()    

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [59]:
xtrain,xtest,ytrain,ytest = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

In [68]:
dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt.fit(xtrain,ytrain)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

In [69]:
dt_pred = dt.predict(xtest)

In [70]:
print(classification_report(ytest, dt_pred))

              precision    recall  f1-score   support

           0       1.00      0.85      0.92   1906322
           1       0.01      0.71      0.01      2464

    accuracy                           0.85   1908786
   macro avg       0.50      0.78      0.47   1908786
weighted avg       1.00      0.85      0.92   1908786



In [71]:
x.columns[dt.feature_importances_.argsort()[-3:]].tolist()

['receiver_funnel', 'receiver_in_deg', 'amount']

In [73]:
# here it is giving false positives. We can try random forest to see if it improves the performance.

In [72]:
rf = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)
rf.fit(xtrain, ytrain)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [75]:
rf.score(xtest,ytest)

0.8582957963857656

In [78]:
rf_pred = rf.predict(xtest)

In [79]:
print(classification_report(ytest, rf_pred))

              precision    recall  f1-score   support

           0       1.00      0.86      0.92   1906322
           1       0.01      0.70      0.01      2464

    accuracy                           0.86   1908786
   macro avg       0.50      0.78      0.47   1908786
weighted avg       1.00      0.86      0.92   1908786

